In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from sam.dataset import generate_dual_task_batch, generate_dual_task_batch_fast, get_distributions

from configurations import load_data, set_font_sizes, create_fig, apply_general_styles, save_fig, make_params_dict
from configurations.plot_config import FONTSIZES

import time

apply_general_styles()

# Test batch generation

In [2]:
vocab_size = 50

# Define device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

P_b, P_u, P_o, P_t = get_distributions(vocab_size, device=device,alpha=0.1,b_type='spiked',u_type='dirichlet',
                                       beta=0.5)

Using device: cuda
Generating spiked bigram distribution with alpha =  0.1  and beta =  0.5
Sampling unigram distribution from a Dirichlet distribution with concentration parameter alpha =  0.1


In [4]:
batch_size = 5
L = 20
K = 3
trigget_set = np.arange(K).tolist()

t0 = time.time()
batch = generate_dual_task_batch_fast(batch_size,L,K,P_b,P_u,P_o,P_t,trigger_set=trigget_set)
t1 = time.time()

# t0 = time.time()
# batch = generate_dual_task_batch(batch_size,L,K,P_b,P_u,P_o,P_t,trigger_set=trigget_set)
# t1 = time.time()

print(f"Batch generated in {t1 - t0:.4f} seconds")
print('sequence')
print(batch['sequence'].shape)

print('trigger_set')
print(batch['trigger_set'].shape)

print('output_set')
print(batch['output_set'].shape)

print('counts')
print(batch['counts'].shape)

print('is_trigg')
print(batch['is_trigg'].shape)

print('mask')
print(batch['mask'].shape)


t1  = 15.3872
t2 = 0.0289

speedup = t1 / t2
print(f"Speedup: {speedup:.2f}x")

Batch generated in 0.0075 seconds
sequence
torch.Size([5, 21])
trigger_set
torch.Size([5, 3])
output_set
torch.Size([5, 3])
counts
torch.Size([5, 20])
is_trigg
torch.Size([5, 20])
mask
torch.Size([5, 20, 20])
Speedup: 532.43x


In [9]:
import json
from dataclasses import dataclass , asdict
from typing import Optional

In [10]:
@dataclass
class ModelArgs:
    vocab_size: int = 64
    seq_len: int = 256
    d_model: int = 128
    dropout: float = 0.0
    K: int = 20

@dataclass
class OptimArgs:
    lr: float = 1e-3
    opt: str = "adam"
    momentum: float = 0.9
    weight_decay: float = 0.0

@dataclass
class DataArgs:
    batch_size: int = 64
    test_size: int = 200

@dataclass
class ExperimentArgs:
    experiment_name: str = "dual_task"
    steps: int = 600
    n_prints: int = 60
    n_prints_model: int = 6
    print_scale: str = "linear"
    init: str = "random"
    b_type: str = "dirichlet"
    u_type: str = "zipf"
    alpha: float = 1.0
    beta: float = 0.5
    fix_trig: bool = True
    trig_type: str = "freq"

@dataclass
class TrainerArgs:
    model_args: ModelArgs
    optim_args: OptimArgs
    data_args: DataArgs
    experiment_args: ExperimentArgs
    
# Initialize full config
trainer_args = TrainerArgs(
    model_args=ModelArgs(),
    optim_args=OptimArgs(),
    data_args=DataArgs(),
    experiment_args=ExperimentArgs()
)

# Convert the dataclass to a dictionary
trainer_args_dict = asdict(trainer_args)

# Save the dictionary to a JSON file
with open(f"../logs/{trainer_args.experiment_args.experiment_name}_config.json", "w") as f:
    json.dump(trainer_args_dict, f, indent=4)

In [7]:
trainer_args.__dict__

{'model_args': ModelArgs(vocab_size=64, seq_len=256, d_model=128, dropout=0.0, K=20),
 'optim_args': OptimArgs(lr=0.001, opt='adam', momentum=0.9, weight_decay=0.0),
 'data_args': DataArgs(batch_size=64, test_size=200),
 'experiment_args': ExperimentArgs(experiment_name='dual_task', steps=600, n_prints=60, n_prints_model=6, print_scale='linear', init='random', b_type='dirichlet', u_type='zipf', alpha=1.0, beta=0.5, fix_trig=True, trig_type='freq')}